<a href="https://colab.research.google.com/github/Balamurugan-T326/DEEP-LEARNING/blob/main/EXPERIMENT-10/PART-A%2CB%2CC%2CD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PART A: Environment Setup and Model Selection

In [11]:
# EXPERIMENT NO: 10
# BUILDING A SIMPLE LLM CHATBOT USING GRADIO
# Student Roll Number: 24BAD016

!pip install -q transformers torch gradio sentencepiece

Load the pretrained model

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("24BAD016")
print("Model:", MODEL_NAME)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
24BAD016
Model: Qwen/Qwen2.5-1.5B-Instruct


PART B: Prompt Handling and Response Generation

In [3]:
def chatbot(message, history):
    if not message or not message.strip():
        return "Please enter a question."

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant for students. "
                "Answer the user's actual question directly and clearly. "
                "Use simple English and provide accurate explanations. "
                "For technical questions, include a short example when useful. "
                "If you are uncertain, say so instead of inventing facts."
            )
        }
    ]

    # Include recent conversation turns
    for item in (history or [])[-6:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")

            if role in ("user", "assistant") and isinstance(content, str):
                if content.strip():
                    messages.append({
                        "role": role,
                        "content": content
                    })

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_msg, bot_msg = item

            if isinstance(user_msg, str) and user_msg.strip():
                messages.append({
                    "role": "user",
                    "content": user_msg
                })

            if isinstance(bot_msg, str) and bot_msg.strip():
                messages.append({
                    "role": "assistant",
                    "content": bot_msg
                })

    messages.append({
        "role": "user",
        "content": message.strip()
    })

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated answer
        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

        answer = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        return answer or "I could not generate an answer. Please try again."

    except Exception as e:
        print("Chatbot error:", repr(e))
        return "An error occurred while generating the answer. Please try again."

Part C:Create the Gradio chatbot interface

In [5]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chatbot,
    title="LLM Chatbot - Experiment 10",
    description="An instruction-following AI chatbot for answering questions.",
    examples=[
        "What is artificial intelligence?",
        "Explain Python programming with an example.",
        "What is the role of a tokenizer in an LLM?",
        "What is the difference between AI and machine learning?",
        "Tell me a short joke."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2289eeea61357c5c39.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
